# U-Net Sanity Check (standalone)

Single-purpose notebook: reload the already-trained U-Net checkpoint (`unet_stage1_baseline.pth`)
and diagnose why its recall (28.28%) is so far below Stage 1 watershed's (66.88% @ IoU0.5, per
Table 5) -- **no retraining**, inference only, so this runs fast.

**Before running:** attach the `After A Rerun` dataset (uploaded from this morning's downloaded
results) as an input to this session -- that's where `unet_stage1_baseline.pth` lives. Everything
below auto-detects it under `/kaggle/input`, no path editing needed.

Run the two cells in order: Setup, then the sanity check.

## Setup -- paths, imports, image-level split (needed to rebuild val_img_names)

In [ ]:
import os, sys, json, time, random, gc
from pathlib import Path
from collections import defaultdict

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as T
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from PIL import Image
from tqdm.auto import tqdm

import matplotlib
import matplotlib.pyplot as plt
# NOT using matplotlib.use("Agg") here -- Agg is file-only and blocks inline
# display, but the whole point of the Plots cell (end of notebook) is that
# figures render AS CELL OUTPUT so they're saved with the notebook itself
# (survives even if /kaggle/working gets wiped before you download it) --
# Kaggle/Jupyter's default backend already supports this, Agg would break it.

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ---- Kaggle paths -- auto-detected, no editing needed ----
# Searches every /kaggle/input/<dataset-slug>/ for a "malaria" folder that
# contains training.json (matches the BBBC041 layout: malaria/images/,
# malaria/training.json, malaria/test.json -- whatever slug Kaggle mounted
# the dataset under).
_candidates = list(Path("/kaggle/input").glob("*/malaria/training.json")) + \
              list(Path("/kaggle/input").glob("*/*/malaria/training.json")) + \
              list(Path("/kaggle/input").glob("malaria/training.json"))
if not _candidates:
    # fall back to a manual path if auto-detect fails -- edit this line
    DATA_ROOT = Path("/kaggle/input/bbbc041-malaria/malaria")
else:
    DATA_ROOT = _candidates[0].parent

TRAIN_JSON = DATA_ROOT / "training.json"
TEST_JSON  = DATA_ROOT / "test.json"
IMG_DIR    = DATA_ROOT / "images"
OUT_DIR    = Path("/kaggle/working/phase6_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_JSON.exists(), (
    f"Not found: {TRAIN_JSON} -- auto-detect failed. Run "
    f"`!find /kaggle/input -maxdepth 3` in a cell to see the actual mounted "
    f"path, then set DATA_ROOT manually above."
)
print("Train JSON :", TRAIN_JSON)
print("Test JSON  :", TEST_JSON, "exists:", TEST_JSON.exists())
print("Image dir  :", IMG_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "--", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")
# ---- shared/label_map.py, inlined ----
LABEL_TO_INT = {
    "background":     0,
    "red blood cell": 1,
    "trophozoite":    2,
    "ring":           3,
    "schizont":       4,
    "gametocyte":     5,
    "leukocyte":      6,
}
INT_TO_LABEL = {v: k for k, v in LABEL_TO_INT.items()}
NUM_CLASSES  = 7
PARASITE_CLASSES = ["trophozoite", "ring", "schizont", "gametocyte"]
SKIP_LABELS = {"difficult"}

print("Classes:", INT_TO_LABEL)

class MalariaDataset(Dataset):
    """Whole-image dataset -- identical to Phase1-EDA/dataset.py. Used here
    only to reproduce the exact same 966/242 (actually 967/241, int() truncation)
    image-level train/val split via the same random_split(seed=42) call, so
    this sanity check evaluates on the identical val-split images as every
    other baseline in the paper."""

    def __init__(self, json_path, image_dir):
        self.image_dir = Path(image_dir)
        self._records = self._parse(Path(json_path))

    def _parse(self, json_path):
        with open(json_path) as f:
            raw = json.load(f)
        records = []
        for item in raw:
            img_name = Path(item["image"]["pathname"]).name
            boxes, labels = [], []
            for obj in item.get("objects", []):
                label = obj["category"]
                if label in SKIP_LABELS or label not in LABEL_TO_INT:
                    continue
                bb = obj["bounding_box"]
                x_min = float(bb["minimum"]["c"]); y_min = float(bb["minimum"]["r"])
                x_max = float(bb["maximum"]["c"]); y_max = float(bb["maximum"]["r"])
                if x_max <= x_min or y_max <= y_min:
                    continue
                boxes.append([x_min, y_min, x_max, y_max])
                labels.append(LABEL_TO_INT[label])
            if boxes:
                records.append({"img_name": img_name, "boxes": boxes, "labels": labels})
        return records

    def __len__(self):
        return len(self._records)

    def __getitem__(self, idx):
        return self._records[idx]


full_img_ds = MalariaDataset(TRAIN_JSON, IMG_DIR)
n_val = int(len(full_img_ds) * 0.2)
n_train = len(full_img_ds) - n_val
train_img_idx, val_img_idx = torch.utils.data.random_split(
    range(len(full_img_ds)), [n_train, n_val], generator=torch.Generator().manual_seed(SEED)
)
train_img_names = {full_img_ds._records[i]["img_name"] for i in train_img_idx.indices}
val_img_names   = {full_img_ds._records[i]["img_name"] for i in val_img_idx.indices}
print(f"Image-level split: {len(train_img_names)} train / {len(val_img_names)} val "
      f"(same seed=42 split as Baseline A/B and Pipeline B, U-Net baseline, everything else)")


---
## U-Net sanity check -- diagnostics + threshold sweep (no retraining)

In [ ]:
# ---- U-Net sanity check: diagnostics + threshold sweep (checkpoint reused, no retraining) ----

# --- Self-contained redefinitions (in case Section C's cells weren't re-run this session) ---
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=32):
        super().__init__()
        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.enc3 = ConvBlock(base * 2, base * 4)
        self.enc4 = ConvBlock(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(base * 8, base * 16)
        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.dec4 = ConvBlock(base * 16, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = ConvBlock(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)
        self.out_conv = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out_conv(d1)

DS_W, DS_H = 512, 384
ORIG_W, ORIG_H = 1600, 1200
SCALE_X, SCALE_Y = DS_W / ORIG_W, DS_H / ORIG_H
MAX_CELL_W, MAX_CELL_H = 220, 220
MIN_AREA_DS = 20

def is_oversized(box):
    x1, y1, x2, y2 = box
    return (x2 - x1) > MAX_CELL_W or (y2 - y1) > MAX_CELL_H

def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

val_records_sanity = [r for r in full_img_ds._records if r["img_name"] in val_img_names]

# --- Load the already-trained checkpoint -- no retraining ---
# Checks /kaggle/working first (same-session case), then auto-detects anywhere
# under /kaggle/input (e.g. if you uploaded this morning's downloaded
# "After A Rerun/phase6_outputs" folder as a new Kaggle dataset this session).
_ckpt_candidates = (
    [OUT_DIR / "unet_stage1_baseline.pth"]
    + list(Path("/kaggle/input").glob("**/unet_stage1_baseline.pth"))
)
_ckpt_path = next((p for p in _ckpt_candidates if p.exists()), None)

if _ckpt_path is None:
    raise FileNotFoundError(
        "unet_stage1_baseline.pth not found in /kaggle/working or anywhere under "
        "/kaggle/input. If you uploaded this morning's downloaded results as a new Kaggle "
        "dataset, double check it's attached to this session (Add Data -> Datasets), or just "
        "re-run Section C's training cell to regenerate it (slower, ~25 epochs)."
    )

unet_model = UNet().to(device)
unet_model.load_state_dict(torch.load(_ckpt_path, map_location=device))
unet_model.eval()
print(f"Loaded checkpoint from {_ckpt_path}")

# --- Cache probability maps ONCE (the expensive step) -- thresholds swept cheaply from this ---
mean_t = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
std_t = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

prob_maps = {}
with torch.no_grad():
    for rec in tqdm(val_records_sanity, desc="Caching U-Net probability maps"):
        bgr = cv2.imread(str(IMG_DIR / rec["img_name"]))
        if bgr is None:
            continue
        img_rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        img_ds = cv2.resize(img_rgb, (DS_W, DS_H))
        img_t = torch.from_numpy(img_ds).permute(2, 0, 1).float().unsqueeze(0) / 255.0
        img_t = ((img_t - mean_t) / std_t).to(device)
        prob = torch.sigmoid(unet_model(img_t))[0, 0].cpu().numpy()
        prob_maps[rec["img_name"]] = prob

# --- Diagnostic: raw model output before any filtering ---
raw_pixel_fractions = [float((p > 0.5).mean()) for p in prob_maps.values()]
raw_component_counts = []
for p in prob_maps.values():
    binary = (p > 0.5).astype(np.uint8)
    n_labels, _, _, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    raw_component_counts.append(n_labels - 1)

avg_gt_per_image = float(np.mean([len(r["boxes"]) for r in val_records_sanity]))

print(f"\n{'='*60}\nU-NET SANITY CHECK -- DIAGNOSTICS (threshold=0.5)\n{'='*60}")
print(f"  Avg predicted positive-pixel fraction : {np.mean(raw_pixel_fractions):.4f}")
print(f"  Avg raw connected components / image  : {np.mean(raw_component_counts):.1f}  "
      f"(before area/oversize filters)")
print(f"  Avg GT boxes / image                  : {avg_gt_per_image:.1f}")
if np.mean(raw_component_counts) < avg_gt_per_image * 0.5:
    print("  -> Model is producing far fewer regions than there are cells -- likely merging "
        "adjacent cells into blobs (undersegmentation) or missing dim/small cells entirely.")

# --- Threshold sweep, reusing cached probability maps (cheap, no re-inference) ---
def boxes_from_prob(prob, threshold, apply_area_filter=True):
    binary = (prob > threshold).astype(np.uint8)
    n_labels, labels_im, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    boxes = []
    for i in range(1, n_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if apply_area_filter and area < MIN_AREA_DS:
            continue
        x, y, w, h = (stats[i, cv2.CC_STAT_LEFT], stats[i, cv2.CC_STAT_TOP],
                      stats[i, cv2.CC_STAT_WIDTH], stats[i, cv2.CC_STAT_HEIGHT])
        x1, y1 = x / SCALE_X, y / SCALE_Y
        x2, y2 = (x + w) / SCALE_X, (y + h) / SCALE_Y
        boxes.append((x1, y1, x2, y2))
    return boxes

def eval_at(threshold, apply_area_filter=True):
    total_gt, total_pred, tp = 0, 0, 0
    for rec in val_records_sanity:
        prob = prob_maps.get(rec["img_name"])
        if prob is None:
            continue
        gt_boxes = [tuple(b) for b in rec["boxes"]]
        pred_boxes = [b for b in boxes_from_prob(prob, threshold, apply_area_filter) if not is_oversized(b)]
        total_gt += len(gt_boxes)
        total_pred += len(pred_boxes)
        gt_matched = [False] * len(gt_boxes)
        for pb in pred_boxes:
            best_iou, best_j = 0.0, -1
            for j, gb in enumerate(gt_boxes):
                if gt_matched[j]:
                    continue
                v = iou(pb, gb)
                if v > best_iou:
                    best_iou, best_j = v, j
            if best_iou >= 0.5 and best_j >= 0:
                gt_matched[best_j] = True
                tp += 1
    recall = tp / total_gt if total_gt else 0.0
    precision = tp / total_pred if total_pred else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"recall": round(recall, 4), "precision": round(precision, 4), "f1": round(f1, 4),
            "n_pred": total_pred, "n_gt": total_gt}

print(f"\n{'='*60}\nTHRESHOLD SWEEP (IoU >= 0.5, area filter ON)\n{'='*60}")
sweep_results = {}
for thr in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    r = eval_at(thr, apply_area_filter=True)
    sweep_results[str(thr)] = r
    print(f"  threshold={thr}: recall={r['recall']:.4f}  precision={r['precision']:.4f}  "
          f"f1={r['f1']:.4f}  (n_pred={r['n_pred']}, n_gt={r['n_gt']})")

best_thr = max(sweep_results, key=lambda k: sweep_results[k]["f1"])
print(f"\nBest threshold by F1: {best_thr} -> {sweep_results[best_thr]}")

# --- Isolate the area filter's own effect at the best threshold ---
no_filter_result = eval_at(float(best_thr), apply_area_filter=False)
print(f"\nSame threshold ({best_thr}) with MIN_AREA_DS filter OFF: {no_filter_result}")
print(f"(compare to WITH filter: {sweep_results[best_thr]} -- a big recall jump here would mean "
      f"MIN_AREA_DS={MIN_AREA_DS} is filtering out real small cells, not just noise)")

sanity_summary = {
    "diagnostics": {
        "avg_positive_pixel_fraction_thr0.5": round(float(np.mean(raw_pixel_fractions)), 4),
        "avg_raw_components_per_image_thr0.5": round(float(np.mean(raw_component_counts)), 2),
        "avg_gt_boxes_per_image": round(avg_gt_per_image, 2),
    },
    "threshold_sweep": sweep_results,
    "best_threshold": best_thr,
    "best_threshold_no_area_filter": no_filter_result,
    "original_result_for_reference": {
        "threshold": 0.5, "area_filter": True,
        "recall": 0.2828, "precision": 0.8135, "f1": 0.4196,
    },
}
with open(OUT_DIR / "unet_sanity_check.json", "w") as f:
    json.dump(sanity_summary, f, indent=2)
print(f"\nSaved -> {OUT_DIR / 'unet_sanity_check.json'}")


---
## Follow-up -- watershed-split post-processing (reuses cached probability maps)

The threshold sweep showed recall rising *with* the threshold (0.24 -> 0.31 from 0.2 -> 0.7)
while precision stayed flat and box count kept growing -- the signature of **merged blobs**
splitting apart as the threshold gets stricter, not of finding more real cells. `MIN_AREA_DS`
was already ruled out as the cause. This cell applies a proper split step to U-Net's own
predicted mask -- distance transform + local-maxima watershed, the same idea Stage 1 already
uses -- instead of relying on a single global threshold to separate touching cells. No
retraining, no re-inference: reuses `prob_maps` already cached in memory from the cell above.


In [ ]:
# ---- Extended threshold sweep (checking if the upward trend keeps climbing) ----
print(f"{'='*60}\nEXTENDED THRESHOLD SWEEP\n{'='*60}")
for thr in [0.75, 0.8, 0.85, 0.9]:
    r = eval_at(thr, apply_area_filter=True)
    print(f"  threshold={thr}: recall={r['recall']:.4f}  precision={r['precision']:.4f}  "
          f"f1={r['f1']:.4f}  (n_pred={r['n_pred']})")

# ---- Watershed-split post-processing on U-Net's own probability map ----
from scipy import ndimage as ndi
try:
    from skimage.feature import peak_local_max
    from skimage.segmentation import watershed as sk_watershed
    _SKIMAGE_OK = True
except ImportError:
    _SKIMAGE_OK = False
    print("scikit-image not available in this environment -- skipping watershed-split test.")

if _SKIMAGE_OK:
    def boxes_from_prob_watershed(prob, threshold=0.5, min_distance=5, min_area=MIN_AREA_DS):
        binary = (prob > threshold).astype(np.uint8)
        if binary.sum() == 0:
            return []
        distance = ndi.distance_transform_edt(binary)
        coords = peak_local_max(distance, min_distance=min_distance, labels=binary)
        if len(coords) == 0:
            return []
        mask = np.zeros(distance.shape, dtype=bool)
        mask[tuple(coords.T)] = True
        markers, _ = ndi.label(mask)
        if markers.max() == 0:
            return []
        labels_ws = sk_watershed(-distance, markers, mask=binary)
        boxes = []
        for lbl in range(1, labels_ws.max() + 1):
            ys, xs = np.where(labels_ws == lbl)
            if len(xs) < min_area:
                continue
            x0, x1b = xs.min(), xs.max() + 1
            y0, y1b = ys.min(), ys.max() + 1
            boxes.append((x0 / SCALE_X, y0 / SCALE_Y, x1b / SCALE_X, y1b / SCALE_Y))
        return boxes

    def eval_watershed_split(threshold, min_distance=5):
        total_gt, total_pred, tp = 0, 0, 0
        for rec in val_records_sanity:
            prob = prob_maps.get(rec["img_name"])
            if prob is None:
                continue
            gt_boxes = [tuple(b) for b in rec["boxes"]]
            pred_boxes = [b for b in boxes_from_prob_watershed(prob, threshold, min_distance)
                          if not is_oversized(b)]
            total_gt += len(gt_boxes)
            total_pred += len(pred_boxes)
            gt_matched = [False] * len(gt_boxes)
            for pb in pred_boxes:
                best_iou, best_j = 0.0, -1
                for j, gb in enumerate(gt_boxes):
                    if gt_matched[j]:
                        continue
                    v = iou(pb, gb)
                    if v > best_iou:
                        best_iou, best_j = v, j
                if best_iou >= 0.5 and best_j >= 0:
                    gt_matched[best_j] = True
                    tp += 1
        recall = tp / total_gt if total_gt else 0.0
        precision = tp / total_pred if total_pred else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        return {"recall": round(recall, 4), "precision": round(precision, 4), "f1": round(f1, 4),
                "n_pred": total_pred, "n_gt": total_gt}

    print(f"\n{'='*60}\nWATERSHED-SPLIT POST-PROCESSING (on U-Net's probability map)\n{'='*60}")
    ws_results = {}
    for thr in [0.4, 0.5, 0.6]:
        for min_dist in [3, 5, 8]:
            r = eval_watershed_split(thr, min_dist)
            key = f"thr={thr}_mindist={min_dist}"
            ws_results[key] = r
            print(f"  {key}: recall={r['recall']:.4f}  precision={r['precision']:.4f}  "
                  f"f1={r['f1']:.4f}  (n_pred={r['n_pred']}, n_gt={r['n_gt']})")

    best_key = max(ws_results, key=lambda k: ws_results[k]["f1"])
    print(f"\nBest combo by F1: {best_key} -> {ws_results[best_key]}")
    print(f"Compare to naive connected-components best (threshold=0.7): "
          f"recall=0.3059, precision=0.8308, f1=0.4472")
    print(f"Compare to Stage 1 watershed (Table 5): recall=0.6688 @ IoU0.5, "
          f"centroid-in-box=0.7904, bio-localization=0.7950")

    with open(OUT_DIR / "unet_watershed_split_sweep.json", "w") as f:
        json.dump(ws_results, f, indent=2)
    print(f"\nSaved -> {OUT_DIR / 'unet_watershed_split_sweep.json'}")


---
## Apples-to-apples fix v2 -- correct per-image resolution

The first version of this cell evaluated on the right *images* (`test.json`, the real 120-image
BBBC041 held-out test set) but the wrong *resolution*: it reused `SCALE_X`/`SCALE_Y`, computed
from `training.json`'s image size (1600x1200). Checked directly -- every one of the 120 test
images is actually **1944x1383px**, a clean, uniform difference from training's 1600x1200 (not a
mixed/occasional thing). Using the training-set scale factor shrank every recovered box toward
the top-left corner, which is why the first run showed catastrophic ~2-5% recall -- that was a
coordinate bug, not U-Net's real test-set performance.

This cell re-runs the same sweeps using each image's actual dimensions (read directly via
`cv2.imread(...).shape`, same approach the production Stage 1 v2/v3 code already uses instead of
a hardcoded constant). Still no retraining -- same checkpoint, same cached-inference approach.


In [ ]:
# ---- APPLES-TO-APPLES FIX v2: correct per-image resolution (test.json is NOT 1600x1200!) ----
# Bug found in the previous version of this cell: it reused the module-level
# SCALE_X/SCALE_Y (= DS_W/1600, DS_H/1200), which is correct for training.json
# images but WRONG for test.json. Checked directly: every one of the 120
# test.json images is 1944x1383px, while every one of the 1208 training.json
# images is 1600x1200px (confirmed by reading all image dimensions, not a
# mixed/occasional thing -- a clean, uniform difference between the two files).
# Using the training-set scale factor on test images shrinks every recovered
# box toward the top-left corner (by ~18% in x, ~13% in y) -- which explains
# why the first run of this cell showed catastrophic 1.7-5% recall: that was
# a coordinate bug, not a real measurement of U-Net's test-set performance.
# Fix: read each image's actual (H, W) at cache time and scale per-image,
# same approach the production Stage 1 v2/v3 code already uses (see
# src/pipeline_b_v2/stage1_v3.py -- resolution_scale(H, W), never hardcoded).

assert TEST_JSON.exists(), f"{TEST_JSON} not found."

test_img_ds = MalariaDataset(TEST_JSON, IMG_DIR)
test_records = test_img_ds._records
print(f"Loaded {len(test_records)} test images, "
      f"{sum(len(r['boxes']) for r in test_records)} GT boxes total "
      f"(paper's Table 5: 120 images, 5,917 GT boxes)")

# --- Cache probability maps AND each image's true (orig_h, orig_w) -- no hardcoded resolution ---
test_prob_maps = {}
test_orig_dims = {}
with torch.no_grad():
    for rec in tqdm(test_records, desc="Caching U-Net probability maps (TEST set, real dims)"):
        bgr = cv2.imread(str(IMG_DIR / rec["img_name"]))
        if bgr is None:
            continue
        orig_h, orig_w = bgr.shape[:2]
        test_orig_dims[rec["img_name"]] = (orig_h, orig_w)
        img_rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        img_ds = cv2.resize(img_rgb, (DS_W, DS_H))
        img_t = torch.from_numpy(img_ds).permute(2, 0, 1).float().unsqueeze(0) / 255.0
        img_t = ((img_t - mean_t) / std_t).to(device)
        prob = torch.sigmoid(unet_model(img_t))[0, 0].cpu().numpy()
        test_prob_maps[rec["img_name"]] = prob

print("Distinct (H, W) seen in test set:", set(test_orig_dims.values()),
      "-- (paper Table 5 images should all be the same single resolution)")

def boxes_from_prob_scaled(prob, threshold, scale_x, scale_y, apply_area_filter=True):
    binary = (prob > threshold).astype(np.uint8)
    n_labels, labels_im, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    boxes = []
    for i in range(1, n_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if apply_area_filter and area < MIN_AREA_DS:
            continue
        x, y, w, h = (stats[i, cv2.CC_STAT_LEFT], stats[i, cv2.CC_STAT_TOP],
                      stats[i, cv2.CC_STAT_WIDTH], stats[i, cv2.CC_STAT_HEIGHT])
        x1, y1 = x / scale_x, y / scale_y
        x2, y2 = (x + w) / scale_x, (y + h) / scale_y
        boxes.append((x1, y1, x2, y2))
    return boxes

def boxes_from_prob_watershed_scaled(prob, scale_x, scale_y, threshold=0.5, min_distance=5, min_area=MIN_AREA_DS):
    binary = (prob > threshold).astype(np.uint8)
    if binary.sum() == 0:
        return []
    distance = ndi.distance_transform_edt(binary)
    coords = peak_local_max(distance, min_distance=min_distance, labels=binary)
    if len(coords) == 0:
        return []
    mask = np.zeros(distance.shape, dtype=bool)
    mask[tuple(coords.T)] = True
    markers, _ = ndi.label(mask)
    if markers.max() == 0:
        return []
    labels_ws = sk_watershed(-distance, markers, mask=binary)
    boxes = []
    for lbl in range(1, labels_ws.max() + 1):
        ys, xs = np.where(labels_ws == lbl)
        if len(xs) < min_area:
            continue
        x0, x1b = xs.min(), xs.max() + 1
        y0, y1b = ys.min(), ys.max() + 1
        boxes.append((x0 / scale_x, y0 / scale_y, x1b / scale_x, y1b / scale_y))
    return boxes

def _scale_for(img_name):
    orig_h, orig_w = test_orig_dims[img_name]
    return DS_W / orig_w, DS_H / orig_h

def eval_at_test(threshold, apply_area_filter=True):
    total_gt, total_pred, tp = 0, 0, 0
    for rec in test_records:
        prob = test_prob_maps.get(rec["img_name"])
        if prob is None:
            continue
        sx, sy = _scale_for(rec["img_name"])
        gt_boxes = [tuple(b) for b in rec["boxes"]]
        pred_boxes = [b for b in boxes_from_prob_scaled(prob, threshold, sx, sy, apply_area_filter)
                      if not is_oversized(b)]
        total_gt += len(gt_boxes)
        total_pred += len(pred_boxes)
        gt_matched = [False] * len(gt_boxes)
        for pb in pred_boxes:
            best_iou, best_j = 0.0, -1
            for j, gb in enumerate(gt_boxes):
                if gt_matched[j]:
                    continue
                v = iou(pb, gb)
                if v > best_iou:
                    best_iou, best_j = v, j
            if best_iou >= 0.5 and best_j >= 0:
                gt_matched[best_j] = True
                tp += 1
    recall = tp / total_gt if total_gt else 0.0
    precision = tp / total_pred if total_pred else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"recall": round(recall, 4), "precision": round(precision, 4), "f1": round(f1, 4),
            "n_pred": total_pred, "n_gt": total_gt}

def eval_watershed_split_test(threshold, min_distance=5):
    total_gt, total_pred, tp = 0, 0, 0
    for rec in test_records:
        prob = test_prob_maps.get(rec["img_name"])
        if prob is None:
            continue
        sx, sy = _scale_for(rec["img_name"])
        gt_boxes = [tuple(b) for b in rec["boxes"]]
        pred_boxes = [b for b in boxes_from_prob_watershed_scaled(prob, sx, sy, threshold, min_distance)
                      if not is_oversized(b)]
        total_gt += len(gt_boxes)
        total_pred += len(pred_boxes)
        gt_matched = [False] * len(gt_boxes)
        for pb in pred_boxes:
            best_iou, best_j = 0.0, -1
            for j, gb in enumerate(gt_boxes):
                if gt_matched[j]:
                    continue
                v = iou(pb, gb)
                if v > best_iou:
                    best_iou, best_j = v, j
            if best_iou >= 0.5 and best_j >= 0:
                gt_matched[best_j] = True
                tp += 1
    recall = tp / total_gt if total_gt else 0.0
    precision = tp / total_pred if total_pred else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"recall": round(recall, 4), "precision": round(precision, 4), "f1": round(f1, 4),
            "n_pred": total_pred, "n_gt": total_gt}

print(f"\n{'='*60}\nTEST-SET (120-image, CORRECT per-image scale) -- NAIVE THRESHOLD SWEEP\n{'='*60}")
test_naive_results = {}
for thr in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    r = eval_at_test(thr, apply_area_filter=True)
    test_naive_results[str(thr)] = r
    print(f"  threshold={thr}: recall={r['recall']:.4f}  precision={r['precision']:.4f}  "
          f"f1={r['f1']:.4f}  (n_pred={r['n_pred']}, n_gt={r['n_gt']})")

print(f"\n{'='*60}\nTEST-SET (120-image, CORRECT per-image scale) -- WATERSHED-SPLIT SWEEP\n{'='*60}")
test_ws_results = {}
for thr in [0.4, 0.5, 0.6]:
    for min_dist in [3, 5, 8]:
        r = eval_watershed_split_test(thr, min_dist)
        key = f"thr={thr}_mindist={min_dist}"
        test_ws_results[key] = r
        print(f"  {key}: recall={r['recall']:.4f}  precision={r['precision']:.4f}  "
              f"f1={r['f1']:.4f}  (n_pred={r['n_pred']}, n_gt={r['n_gt']})")

best_naive_key = max(test_naive_results, key=lambda k: test_naive_results[k]["f1"])
best_test_key = max(test_ws_results, key=lambda k: test_ws_results[k]["f1"])
print(f"\nBest naive-threshold (TEST set): thr={best_naive_key} -> {test_naive_results[best_naive_key]}")
print(f"Best watershed-split combo (TEST set): {best_test_key} -> {test_ws_results[best_test_key]}")
print(f"\nDIRECT COMPARISON -- same 120-image test.json set as Table 5:")
print(f"  Stage 1 watershed (Table 5)      : recall=0.6688, centroid-in-box=0.7904, bio-localization=0.7950")
print(f"  U-Net naive threshold (best F1)  : {test_naive_results[best_naive_key]}")
print(f"  U-Net + watershed-split (best F1): {test_ws_results[best_test_key]}")

test_set_summary = {
    "note": "Evaluated on the REAL 120-image test.json set (5,917 GT boxes) using each "
            "image's ACTUAL resolution (1944x1383, not the training-set's 1600x1200) -- "
            "the earlier version of this cell used the wrong hardcoded resolution and "
            "produced meaningless near-zero numbers.",
    "n_test_images": len(test_records),
    "n_gt_boxes": sum(len(r["boxes"]) for r in test_records),
    "test_image_dims_seen": list(set(test_orig_dims.values())),
    "naive_threshold_sweep": test_naive_results,
    "watershed_split_sweep": test_ws_results,
    "best_watershed_split_combo": best_test_key,
    "stage1_watershed_table5_reference": {
        "recall_iou50": 0.6688, "centroid_in_box": 0.7904,
        "bio_localization": 0.7950, "infected_sensitivity_iou50": 0.6634,
    },
}
with open(OUT_DIR / "unet_test_set_apples_to_apples.json", "w") as f:
    json.dump(test_set_summary, f, indent=2)
print(f"\nSaved -> {OUT_DIR / 'unet_test_set_apples_to_apples.json'}")
